<a href="https://colab.research.google.com/github/Miaxrz/ML-Internship-Starter/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Miaxrz/ML-Internship-Starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## 1. Unit of analysis + time window

For my lane, Refresh / Content Opportunity Scoring, one row in my final modelling frame represents one content item for one client (`client_hash_id × content_hash_id`).

The raw warehouse table, `fact_content_daily_performance`, is at a finer grain of one client × one content item × one day. I aggregate these daily records into monthly windows for modelling.

I use `fact_content_daily_performance` as my main table.

My feature window is February 2026, and my outcome window is March 2026. The decision point is the end of February 2026, so all features must be information that was available on or before that point.

I will rank pages by their risk of losing search clicks. For this exercise, I use a binary outcome called `went_dark`. A page receives a value of 1 if it has measured GSC data during March 2026 but records zero clicks. A page receives a value of 0 if it has measured GSC data and records one or more clicks.

Pages with no measured GSC data during March are excluded because unavailable data is not the same as zero traffic.

I deliberately exclude March performance metrics from my feature set because March occurs after my February decision point. Using March information as a feature would create data leakage.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import duckdb
import pandas as pd
import numpy as np
import getpass
import os

con = duckdb.connect()

HF_TOKEN = getpass.getpass("Enter your Hugging Face token: ")

con.execute(f"""
    CREATE OR REPLACE SECRET hf_token (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    );
""")


REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

print("Warehouse connection setup complete.")

Enter your Hugging Face token: ··········
Warehouse connection setup complete.


In [13]:
con.sql(f"""
    SELECT *
    FROM {MAR}
    LIMIT 5
""").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [14]:
schema_check = con.sql(f"""
    DESCRIBE SELECT *
    FROM {MAR}
""").df()

display(schema_check)

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 2. Fields: feature / label / context / excluded

### Features

I will use five features from February 2026:

1. `gsc_impressions_feb` — Knowable at the decision moment because GSC impressions were observed during the completed February feature window.

2. `gsc_clicks_feb` — Knowable at the decision moment because GSC clicks were observed during the completed February feature window.

3. `ctr_feb` — Knowable at the decision moment because it is calculated from February GSC clicks and impressions only.

4. `gsc_avg_position_feb` — Knowable at the decision moment because average search position is calculated from GSC observations available during February.

5. `ga4_sessions_feb` — Knowable at the decision moment because GA4 sessions are observed during the completed February feature window.

### Label

The label is `went_dark`.

It is created from March 2026, after the February decision point.

`went_dark = 1` when GSC data is available during March but total GSC clicks are zero.

`went_dark = 0` when GSC data is available during March and total GSC clicks are greater than zero.

### Context

`client_hash_id` and `content_hash_id` identify the client and content item and are used for grouping and joining. They are not predictive features.

### Excluded

I deliberately exclude March performance variables from the feature set because they occur after the February decision point. I also exclude any variable directly derived from the March label.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Define the data contract buckets

feature_columns = [
    "gsc_impressions_feb",
    "gsc_clicks_feb",
    "ctr_feb",
    "gsc_avg_position_feb",
    "ga4_sessions_feb"
]

context_columns = [
    "client_hash_id",
    "content_hash_id"
]

label_column = "went_dark"

excluded_columns = [
    "March performance metrics",
    "Any variable derived from went_dark"
]

print("Features:")
for col in feature_columns:
    print("-", col)

print("\nContext:")
for col in context_columns:
    print("-", col)

print("\nLabel:")
print("-", label_column)

print("\nExcluded:")
for col in excluded_columns:
    print("-", col)

Features:
- gsc_impressions_feb
- gsc_clicks_feb
- ctr_feb
- gsc_avg_position_feb
- ga4_sessions_feb

Context:
- client_hash_id
- content_hash_id

Label:
- went_dark

Excluded:
- March performance metrics
- Any variable derived from went_dark


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*


I will use the March 2026 partition to verify three facts:

1. Grain — whether one raw row corresponds to one `client_hash_id × content_hash_id × report_date`.
2. Slice size and date span — the number of rows and the actual reporting-date range in March 2026.
3. Availability — the number of rows where `gsc_data_available IS TRUE`.

The March partition is selected using `month=2026-03`, while `report_date` is used to verify the actual date range.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ============================================================
# 1. GRAIN CHECK
# ============================================================

grain_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(DISTINCT
            client_hash_id || '|' ||
            content_hash_id || '|' ||
            CAST(report_date AS VARCHAR)
        ) AS unique_client_content_dates

    FROM {MAR}
""").df()

print("GRAIN CHECK")
display(grain_check)


# ============================================================
# 2. ROW COUNT AND DATE SPAN
# ============================================================

slice_check = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date,
        COUNT(DISTINCT client_hash_id) AS unique_clients,
        COUNT(DISTINCT content_hash_id) AS unique_content_items

    FROM {MAR}
""").df()

print("MARCH 2026 SLICE CHECK")
display(slice_check)


# ============================================================
# 3. GSC AVAILABILITY
# ============================================================

availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows

    FROM {MAR}
""").df()

print("GSC AVAILABILITY CHECK")
display(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

GRAIN CHECK


,total_rows,unique_client_content_dates
0,9841378,9841378


MARCH 2026 SLICE CHECK


,row_count,first_date,last_date,unique_clients,unique_content_items
0,9841378,2026-03-01,2026-03-31,55,331437


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

GSC AVAILABILITY CHECK


,total_rows,gsc_available_rows
0,9841378,3611061


### Interpretation of the verification

The grain check compares the total number of rows with the number of unique `client_hash_id × content_hash_id × report_date` combinations. Matching values support the stated raw-row grain.

The March 2026 slice check reports the number of rows and the actual minimum and maximum `report_date`.

The availability check uses `gsc_data_available IS TRUE`, so only explicitly available GSC data is counted as available. Missing or unavailable GSC data is not treated as zero performance.

In [17]:
features_feb = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions)
        FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_impressions_feb,

        SUM(gsc_clicks)
        FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_clicks_feb,

        CASE
            WHEN SUM(gsc_impressions)
                FILTER (
                    WHERE gsc_data_available IS TRUE
                ) > 0

            THEN
                SUM(gsc_clicks)
                FILTER (
                    WHERE gsc_data_available IS TRUE
                ) * 1.0
                /
                SUM(gsc_impressions)
                FILTER (
                    WHERE gsc_data_available IS TRUE
                )

            ELSE NULL
        END AS ctr_feb,

        AVG(gsc_avg_position)
        FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_avg_position_feb,

        SUM(ga4_sessions)
        FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_sessions_feb

    FROM {FEB}

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

print("FEBRUARY FEATURE FRAME")
display(features_feb.head())

print("Shape:", features_feb.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FEBRUARY FEATURE FRAME


,client_hash_id,content_hash_id,gsc_impressions_feb,gsc_clicks_feb,ctr_feb,gsc_avg_position_feb,ga4_sessions_feb
0,client_e547b89c05043229,content_7995404695ee1ffd,1012.0,3.0,0.002964,29.609070,5.0
1,client_e547b89c05043229,content_ccbb253f142217c3,1598.0,7.0,0.004380,17.806923,13.0
2,client_e547b89c05043229,content_ae16a6b9cf64c80a,861.0,0.0,0.000000,7.852945,1.0
3,client_e547b89c05043229,content_acf700633f016e5a,245.0,0.0,0.000000,7.123694,1.0
4,client_e547b89c05043229,content_712e44562fed8ff2,306.0,0.0,0.000000,7.965842,NaN


Shape: (321546, 7)


## 4. Create the March outcome

March is used only to create the future outcome.

I first require at least one March row with `gsc_data_available IS TRUE`.

For those content items:

- `went_dark = 1` when March GSC clicks equal zero.
- `went_dark = 0` when March GSC clicks are greater than zero.

This keeps the future outcome separate from the February feature window.

In [18]:
label_mar = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows_mar,

        SUM(gsc_clicks)
        FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_clicks_mar

    FROM {MAR}

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()


# Keep only content items with measured GSC data
label_mar = label_mar[
    label_mar["gsc_available_rows_mar"] > 0
].copy()


# Missing clicks within available GSC rows are treated as zero
label_mar["gsc_clicks_mar"] = (
    label_mar["gsc_clicks_mar"]
    .fillna(0)
)


# Create future label
label_mar["went_dark"] = (
    label_mar["gsc_clicks_mar"] == 0
).astype(int)


print("MARCH OUTCOME")
display(label_mar.head())

print("\nOutcome distribution:")
display(label_mar["went_dark"].value_counts())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

MARCH OUTCOME


,client_hash_id,content_hash_id,gsc_available_rows_mar,gsc_clicks_mar,went_dark
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,24,0.0,1
1,client_62f4a7e64f5e0096,content_cec711b02f3bbde6,29,4.0,0
2,client_62f4a7e64f5e0096,content_275b6f7f733016d4,29,1.0,0
3,client_62f4a7e64f5e0096,content_ceaec531566ffcfc,27,0.0,1
4,client_62f4a7e64f5e0096,content_755d951187fcd70a,30,6.0,0



Outcome distribution:


,count
went_dark,
1,107901
0,68837


In [19]:
frame = features_feb.merge(
    label_mar[
        [
            "client_hash_id",
            "content_hash_id",
            "went_dark"
        ]
    ],
    on=[
        "client_hash_id",
        "content_hash_id"
    ],
    how="inner"
)

print("FINAL MODELLING FRAME")
print("Shape:", frame.shape)

display(frame.head())

print("\nLabel distribution:")
display(frame["went_dark"].value_counts())

FINAL MODELLING FRAME
Shape: (161539, 8)


,client_hash_id,content_hash_id,gsc_impressions_feb,gsc_clicks_feb,ctr_feb,gsc_avg_position_feb,ga4_sessions_feb,went_dark
0,client_e547b89c05043229,content_7995404695ee1ffd,1012.0,3.0,0.002964,29.609070,5.0,0
1,client_e547b89c05043229,content_ccbb253f142217c3,1598.0,7.0,0.004380,17.806923,13.0,0
2,client_e547b89c05043229,content_ae16a6b9cf64c80a,861.0,0.0,0.000000,7.852945,1.0,1
3,client_e547b89c05043229,content_acf700633f016e5a,245.0,0.0,0.000000,7.123694,1.0,1
4,client_e547b89c05043229,content_712e44562fed8ff2,306.0,0.0,0.000000,7.965842,NaN,0



Label distribution:


,count
went_dark,
1,98368
0,63171


In [20]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

feature_cols = [
    "gsc_impressions_feb",
    "gsc_clicks_feb",
    "ctr_feb",
    "gsc_avg_position_feb",
    "ga4_sessions_feb"
]

model_df = frame.dropna(
    subset=feature_cols + ["went_dark"]
).copy()

X = model_df[feature_cols]
y = model_df["went_dark"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

honest_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

honest_model.fit(X_train, y_train)

honest_scores = honest_model.predict_proba(
    X_test
)[:, 1]

honest_auc = roc_auc_score(
    y_test,
    honest_scores
)

print("Honest ROC-AUC:", round(honest_auc, 4))

Honest ROC-AUC: 0.8536


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

The data has several limitations.

First, GSC and GA4 availability is not guaranteed for every row. I therefore use the explicit availability flags rather than assuming that missing data represents zero performance.

Second, the February feature window and March outcome window must remain separate. March performance cannot be used as a February feature because it would not have been available at the decision point.

Third, the `went_dark` outcome is a proxy for prioritisation rather than proof of why a content item lost traffic. A zero-click month does not by itself establish the cause of the change.

Finally, the leakage experiment demonstrates that unrealistic information can produce an artificially strong model score. Therefore, feature availability and time ordering must be checked before trusting model performance.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Check important limitations in the starter dataset


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.